## Pipeline di caricamento dati su SQL
In questo notebook configuriamo una pipeline per il caricamento massivo dei dati nel database MariaDB. Iniziamo importando le librerie necessarie per la manipolazione dei dati (`pandas`) e per la gestione della connessione al database (`sqlalchemy`).



In [103]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

### Caricamento del Dataset Pulito
Ricarichiamo il file CSV precedentemente pulito e processato nella fase di EDA. Questo file contiene tutte le informazioni necessarie, ma è ancora in formato "flat" (piatto), privo degli identificativi numerici necessari per lo schema relazionale SQL.

In [104]:
url= r"C:\Users\angel\OneDrive\Desktop\Lavoro STAFF\hotel_reviews_clean.csv"
hr1= pd.read_csv(url)
engine = create_engine('mysql+mysqlconnector://root:@localhost/Hotel_Reviews')

# Modifica il blocco della connessione così:
with engine.connect() as conn:
    # 1. Disabilita i controlli
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    
    # 2. Cancella le tabelle
    conn.execute(text("DROP TABLE IF EXISTS hotel_stats;"))
    conn.execute(text("DROP TABLE IF EXISTS reviewers;"))
    conn.execute(text("DROP TABLE IF EXISTS hotels;"))
    conn.execute(text("DROP TABLE IF EXISTS reviews;"))

    conn.commit()
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    print("Database pulito. Ora procedi con il caricamento Pandas...")
 

Database pulito. Ora procedi con il caricamento Pandas...


### Popolamento delle Anagrafiche (Hotels e Reviewers)

In questa fase iniziamo a popolare le tabelle "anagrafiche" del database, che fungeranno da tabelle padre per le relazioni successive.

### Logica dell'operazione:
1. **Estrazione Dati Univoci**: Utilizziamo il metodo `.drop_duplicates()` per assicurarci di caricare ogni Hotel e ogni Nazionalità una sola volta. Questo garantisce l'integrità dei dati ed evita errori di duplicazione sulle Primary Keys.
2. **Rinomina Colonne**: Uniformiamo i nomi delle colonne di Pandas a quelli definiti nello schema SQL (es. da `Hotel_Name` a `hotel_name`).
3. **Caricamento Massivo**: Utilizziamo `to_sql` con l'opzione `append` per popolare le tabelle già configurate su phpMyAdmin. In questo modo evitiamo di sovrascrivere la struttura del database, garantendo che i dati vengano inseriti correttamente rispettando le relazioni e i formati (come `DECIMAL` o `TEXT`) stabiliti durante la fase di progettazione.


Questa procedura è fondamentale per generare gli ID numerici univoci (Auto-increment) che useremo in seguito per collegare le recensioni e le statistiche.


In [105]:
# POPOLAMENTO TABELLA HOTELS 
# Prendo solo un hotel per ogni nome, ignorando micro-differenze negli indirizzi
hotels = hr1[['Hotel_Name', 'Hotel_Address', 'lat', 'lng']].drop_duplicates(subset=['Hotel_Name', 'Hotel_Address'])
hotels.columns = ['hotel_name', 'hotel_address', 'lat', 'lng']
hotels.insert(0, 'hotel_id', range(1, len(hotels) + 1))
hotels.to_sql('hotels', con=engine, if_exists='replace', index=False)

# POPOLAMENTO TABELLA REVIEWERS 
reviewers = hr1[['Reviewer_Nationality']].drop_duplicates()
reviewers.columns = ['reviewer_nationality']
reviewers.insert(0, 'reviewer_id', range(1, len(reviewers) + 1))
reviewers.to_sql('reviewers', con=engine, if_exists='replace', index=False)

print("Tabelle Hotels e Reviewers popolate con ID manuali.")

Tabelle Hotels e Reviewers popolate con ID manuali.


### Sincronizzazione degli ID (Join tra Python e SQL)
Dopo aver popolato le anagrafiche, interroghiamo il database per recuperare i codici identificativi appena generati. Questi ID sono il "ponte" necessario per collegare ogni singola recensione al suo hotel e al suo autore, assicurando che l'intero sistema relazionale sia coerente e solido.



In [106]:
sql_hotels = pd.read_sql("SELECT hotel_id, hotel_name, hotel_address FROM hotels", engine)
sql_reviewers = pd.read_sql("SELECT reviewer_id, reviewer_nationality FROM reviewers", engine)


### Popolamento della tabella Hotel_Stats (Gestione Duplicati)

In questa fase popoliamo la tabella delle statistiche aggregate. Poiché la tabella `hotel_stats` utilizza `hotel_id` come **Chiave Primaria**, è fondamentale che ogni hotel compaia esattamente una sola volta.

### Passaggi chiave:

1. **Mapping degli identificativi**: Associamo i nomi degli hotel presenti nel dataset ai rispettivi codici numerici (`hotel_id`) recuperati dal database.
2. **Consolidamento dei dati**: Poiché il dataset originale può contenere micro-variazioni nei punteggi per lo stesso hotel, utilizziamo il metodo `.groupby().agg('max')`. Questo garantisce la creazione di un record unico e coerente per ogni struttura.
3. **Gestione delle relazioni 1:1**: Questa procedura evita conflitti con la Chiave Primaria su SQL e assicura che ogni hotel sia collegato correttamente alle proprie statistiche globali senza generare ridondanze.



In [107]:
# POPOLAMENTO TABELLA HOTEL_STATS 
# 1. Filtriamo solo le colonne necessarie
stats = hr1[['Hotel_Name', 'Average_Score', 'Total_Number_of_Reviews', 'Additional_Number_of_Scoring']]

# 2. Colleghiamo gli ID degli hotel
stats = stats.merge(sql_hotels, left_on='Hotel_Name', right_on='hotel_name')

# 3. Gestiamo i duplicati aggregando: prendiamo il valore massimo per ogni ID hotel
# Questo risolve eventuali errori della chiave primaria
stats = stats.groupby('hotel_id').agg({
    'Average_Score': 'max',
    'Total_Number_of_Reviews': 'max',
    'Additional_Number_of_Scoring': 'max'
}).reset_index()

# 4. Rinominiamo le colonne per SQL
stats.columns = ['hotel_id', 'average_score', 'total_reviews', 'additional_number_of_scoring']

# 5. Carichiamo
stats.to_sql('hotel_stats', con=engine, if_exists='replace', index=False)

print("Tabella Hotel_Stats popolata correttamente senza duplicati.")


Tabella Hotel_Stats popolata correttamente senza duplicati.


## Caricamento Massivo della Tabella 'Reviews' (Bulk Insert)

Questo passaggio finale è dedicato esclusivamente al popolamento della tabella `reviews`. Data la presenza di oltre 511.000 record e la complessità dei testi contenuti nelle recensioni, ho deciso di  implementare una strategia di caricamento ottimizzata.

### Dettagli tecnici del processo:
1. **Mapping Finale degli ID**: Colleghiamo l'intero dataset originale agli ID univoci degli Hotel e dei Recensori per stabilire le relazioni (Foreign Keys) richieste dallo schema SQL.
2. **Selezione e Formattazione**: Filtriamo solo le colonne necessarie e le rinominiamo per farle coincidere esattamente con la struttura della tabella di destinazione.
3. **Tecnica di Chunking**: Utilizziamo il parametro `chunksize=10000` per inviare i dati a blocchi. Questo previene il sovraccarico della memoria RAM e scongiura errori di timeout del database MariaDB, garantendo un caricamento fluido ed efficiente.

Conclusa questa fase, l'intero ecosistema relazionale del database è popolato e pronto per le interrogazioni analitiche.



In [108]:
# Uniamo gli ID al dataframe originale per creare i collegamenti (Foreign Key)
hr_final = hr1.merge(sql_hotels, 
                     left_on=['Hotel_Name', 'Hotel_Address'], 
                     right_on=['hotel_name', 'hotel_address'], 
                     how='inner')

hr_final = hr_final.merge(sql_reviewers, 
                          left_on='Reviewer_Nationality', 
                          right_on='reviewer_nationality')
# Selezioniamo e rinominiamo le colonne per farle combaciare con lo schema SQL
reviews_table = hr_final[[
    'hotel_id', 'reviewer_id', 'Review_Date', 'Reviewer_Score', 
    'Negative_Review', 'Positive_Review', 'Review_Total_Negative_Word_Counts', 
    'Review_Total_Positive_Word_Counts', 'Total_Number_of_Reviews_Reviewer_Has_Given',
    'days_since_review', 'Tags', 'Review_Year', 'Review_Month', 
    'Days_Since_Numeric', 'Total_Review_Length'
]]

reviews_table.columns = [
    'hotel_id', 'reviewer_id', 'review_date', 'reviewer_score', 
    'negative_review', 'positive_review', 'review_total_negative_word_counts', 
    'review_total_positive_word_counts', 'total_reviews_reviewer_has_given',
    'days_since_review', 'tags', 'review_year', 'review_month', 
    'days_since_numeric', 'total_review_length'
]

#Inserisco l'id autoincrementante per le reviews
reviews_table.insert(0, 'review_id', range(1, len(reviews_table) + 1))

# Caricamento massivo (circa 515.000 righe)
reviews_table.to_sql('reviews', con=engine, if_exists='replace', index=False, chunksize=10000)

print("Dati caricati con review_id incluso! Inizio configurazione vincoli e chiavi...")



Dati caricati con review_id incluso! Inizio configurazione vincoli e chiavi...


In [ ]:
with engine.connect() as conn:
    # 1. Disabilitiamo i controlli per poter modificare le colonne liberamente
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))

    # 2. Uniformiamo i tipi di dato (Forziamo tutto a INT) e impostiamo le Primary Keys
    # Tabella HOTELS
    conn.execute(text("""
    ALTER TABLE hotels 
    MODIFY hotel_name VARCHAR(255),
    MODIFY hotel_address VARCHAR(500),
    MODIFY lat DECIMAL(10, 8),
    MODIFY lng DECIMAL(11, 8),
    MODIFY hotel_id INT NOT NULL;
"""))
    try: conn.execute(text("ALTER TABLE hotels DROP PRIMARY KEY;"))
    except: pass
    conn.execute(text("ALTER TABLE hotels ADD PRIMARY KEY (hotel_id);"))
    
    # Tabella REVIEWERS
    conn.execute(text("""
    ALTER TABLE reviewers 
    MODIFY reviewer_nationality VARCHAR(100),
    MODIFY reviewer_id INT NOT NULL;
"""))
    try: conn.execute(text("ALTER TABLE reviewers DROP PRIMARY KEY;"))
    except: pass
    conn.execute(text("ALTER TABLE reviewers ADD PRIMARY KEY (reviewer_id);"))
    
    # Tabella HOTEL_STATS
    conn.execute(text("""
        ALTER TABLE hotel_stats 
        MODIFY average_score DECIMAL(3, 1),
        MODIFY hotel_id INT NOT NULL PRIMARY KEY;
    """))
    try: conn.execute(text("ALTER TABLE hotel_stats DROP PRIMARY KEY;"))
    except: pass
    conn.execute(text("ALTER TABLE hotel_stats ADD PRIMARY KEY (hotel_id);"))
    
    # Tabella REVIEWS 
    conn.execute(text("""
    ALTER TABLE reviews 
    MODIFY review_id INT NOT NULL,
    MODIFY hotel_id INT NOT NULL,
    MODIFY reviewer_id INT NOT NULL,
    MODIFY review_date DATE,
    MODIFY negative_review TEXT,
    MODIFY positive_review TEXT,
    MODIFY reviewer_score DECIMAL(3,1),
    MODIFY total_review_length INT,
    MODIFY days_since_review VARCHAR(50),
    MODIFY tags VARCHAR(500)   ,
    MODIFY review_year INT,
    MODIFY review_month INT,
    MODIFY days_since_numeric INT,
    MODIFY review_total_negative_word_counts INT,
    MODIFY review_total_positive_word_counts INT,
    MODIFY total_reviews_reviewer_has_given INT;
"""))

    try: conn.execute(text("ALTER TABLE reviews DROP PRIMARY KEY;"))
    except: pass
    conn.execute(text("ALTER TABLE reviews ADD PRIMARY KEY (review_id);"))

    # 3. CREAZIONE DEI VINCOLI 
    conn.execute(text("""
        ALTER TABLE reviews 
        ADD CONSTRAINT fk_reviews_hotels 
        FOREIGN KEY (hotel_id) REFERENCES hotels(hotel_id);
    """))
    conn.execute(text("""
        ALTER TABLE reviews 
        ADD CONSTRAINT fk_reviews_reviewers 
        FOREIGN KEY (reviewer_id) REFERENCES reviewers(reviewer_id);
    """))
    conn.execute(text("""
        ALTER TABLE hotel_stats 
        ADD CONSTRAINT fk_stats_hotels 
        FOREIGN KEY (hotel_id) REFERENCES hotels(hotel_id);
    """))

    # 4. Riabilitiamo tutto
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    conn.commit()

print("DATABASE STRUTTURATO: Tutte le FK sono state create correttamente.")

DATABASE STRUTTURATO: Tutte le FK sono state create correttamente.
